# Lecture 32: Support Vector Machines

CMSE 381, Fall 2026 (Section 002), Nov 16, 2026

![Diagram of a kernel machine: input data mapped through a kernel into a higher-dimensional space where the classes can be separated](https://upload.wikimedia.org/wikipedia/commons/thumb/f/fe/Kernel_Machine.svg/1920px-Kernel_Machine.svg.png)

In [ ]:
# Everyone's favorite standard imports
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
%matplotlib inline
import time


# ML imports we've used previously
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GridSearchCV

# For today, we just need SVC
from sklearn.svm import SVC 

We now have discussed three related methods in class. The book's definitions as we talked about in class are:

- *Maximal Margin Classifiers*, where the goal was to find a separating hyperplane with no misclassifications, 
- *Support vector classifiers*, where we allow for a soft margin and hence some misclassifications, but only allow for a linear kernel, and 
- *Support vector machines*, where we have a soft margin and an option for kernels. 

It turns out that `sklearn` has only one function to do all of this. A reminder from last time. **There are two things that will likely be confusing.**
- The command is just called `SVC`, but you should thinking of it as doing the most general SVM as defined in the book and then we can modify our inputs to allow for the other options as necessary.
- The cost input parameter is not the same as the `C` defined in the book. However, it controls the same thing; that is, the amount of tolerance we have for data points on the wrong side of the margin and/or wrong side of the boundary.

The command below is the same as from the last notebook.  The goal is just to be able to draw the boundaries from the SVM easily. 

In [ ]:
# Run this cell to define the function
def plot_svc(svc, X, y, h=0.02, pad=0.25):
    x_min, x_max = X[:, 0].min()-pad, X[:, 0].max()+pad
    y_min, y_max = X[:, 1].min()-pad, X[:, 1].max()+pad
    xvec = np.arange(x_min, x_max, h)
    yvec = np.arange(y_min, y_max, h)
    xx, yy = np.meshgrid(xvec,yvec )
    
    Z = svc.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    plt.contourf(xx, yy, Z, cmap=plt.cm.Paired, alpha=0.2)

    plt.scatter(X[:,0], X[:,1], s=70, c=y, cmap=mpl.cm.Paired)
    # Support vectors indicated in plot by X's
    sv = svc.support_vectors_
    plt.scatter(sv[:,0], sv[:,1], c='k', marker='x', s=100, linewidths=1)
    
    if svc.kernel == 'linear':
        # Get the margin lines 
        w = svc.coef_[0]
        a = -w[0] / w[1]
        yhyperplane = a * xvec - (svc.intercept_[0]) / w[1]
        margin = 1 / np.sqrt(np.sum(svc.coef_ ** 2))
        ymargin_down = yhyperplane+  - np.sqrt(1 + a ** 2) * margin
        ymargin_up = yhyperplane + np.sqrt(1 + a ** 2) * margin
        plt.plot(xvec,ymargin_down, "k--")
        plt.plot(xvec,ymargin_up, "k--")

    
    
    plt.xlim(x_min, x_max)
    plt.ylim(y_min, y_max)
    plt.xlabel('X1')
    plt.ylabel('X2')
    plt.show()
    print('Number of support vectors: ', svc.support_.size)

# Swapping out the kernel 

In today's class, we've been discussing changing the kernel function and then learning the model
$$
f(x) = \beta_0 + \sum_{i \in \mathcal{S}}\alpha_i K(x,x_i)
$$

In [ ]:
url = "https://lianzhangbao.github.io/CMSE381-Fall2026-Public/DataSets/SVM-Data3.csv"
data3 = np.loadtxt(url)
X = data3[:,:2]
y = data3[:,2]


plt.scatter(X[:,0], X[:,1], s=70, c=y, cmap=mpl.cm.Paired)
plt.xlabel('X1')
plt.ylabel('X2')
plt.show()

**Do this:** Train a SVC using a radial kernel (this is `kernel = 'rbf'` as input to the `SVC` function) and with $C=1$, $\gamma = 1$. Use the `plot_svc` function to see what the learned boundary looks like. 

In [ ]:
# Your code here

**Do this:** What happens if you increase $C$ to 100? Is this model looking better or worse than what you had before?

In [ ]:
# Your code here #

*Your answer here*

**Do this:** Use the `GridSearchCV` function (see the last lab for examples of using it) to determine the best $C$ and $\gamma$ parameters. 

In [ ]:
# Your code here

**Do this:** Use the `plot_svc` function to take a look at the above result.

In [ ]:
##YOUR CODE HERE

# Still have time? 

Download the NIST data set from here: https://archive.ics.uci.edu/ml/datasets/optical+recognition+of+handwritten+digits

You just need two files for now, the training set `optdigits.tra` and the testing set `optdigits.tes`. The following commands will pull the data from the remote server. Optionally, you can download the files directly to your computer.

In [ ]:
X_train = pd.read_csv('https://archive.ics.uci.edu/ml/machine-learning-databases/optdigits/optdigits.tra', header=None)
X_train.head()

In [ ]:
X_test = pd.read_csv('https://archive.ics.uci.edu/ml/machine-learning-databases/optdigits/optdigits.tes', header=None)
X_test.head()

In [ ]:
# X_train = pd.read_csv('optdigits.tra', header=None) #<-- put this line back in if you saved the data locally
y_train = X_train[64]
X_train = X_train.drop(X_train.columns[64], axis=1)

# X_test = pd.read_csv('optdigits.tes', header=None) #<-- put this line back in if you saved the data locally
y_test = X_test[64]
X_test = X_test.drop(X_test.columns[64], axis=1)

In [ ]:
print(X_train.shape)
print(X_test.shape)

This data set consists of 8x8 images of handwritten digits. The following command will draw each data point for you. Mess around with the value of $i$ below to see other examples

In [ ]:
i = 13
plt.imshow(X_train.values[i].reshape(8,8), cmap="gray") 
plt.show()
print(f'Data point {i} is labeled as {y_train[i]}')

The plot above is a small grayscale image (8 by 8 pixels) of one handwritten digit. Its label is printed below the image.

**Do this:** Build a classifier to predict the correct digit for a given handwritten digit. As you do this, answer the following questions:
- What choice of kernel does best? 
- What are the optimal choices of parameters for the SVC? 
- How well does your classifier do? Don't forget that quality measures should always use testing data.

*Your answer here*

**Do this:** 
- How well does your above classifier do? Don't forget that quality measures should always use testing data.

In [ ]:
##YOUR CODE HERE###



-----
## Congratulations, we're done!
Initially created by Dr. Liz Munch, modified by Dr. Lianzhang Bao, Michigan State University

<a rel="license" href="http://creativecommons.org/licenses/by-nc/4.0/"><img alt="Creative Commons License" style="border-width:0" src="https://i.creativecommons.org/l/by-nc/4.0/88x31.png" /></a><br />This work is licensed under a <a rel="license" href="http://creativecommons.org/licenses/by-nc/4.0/">Creative Commons Attribution-NonCommercial 4.0 International License</a>.